##### Import libraries #####

In [98]:
from pathlib import Path
import os
import sys
import SimpleITK as sitk
import resmip as rsm
from Densitometry.dcm_functions import analyze_dcm as info_dcm
from Densitometry.total_ROI_functions import analyze_spacing as sp
from Densitometry.total_ROI_functions import analyze_ROI as ROI
from Densitometry.total_ROI_functions import total as tot
import numpy as np

## **No resample extraction** ##

##### Define paths and directories #####

In [99]:
#Set the directory dcm out and directory out
directory_dcm_out = Path.home() / "Desktop" / "Densitometry" / "tutorials"/"tutorial_patient"/"IBSI1_CT_phantom"

    
directory_out_no_resample = Path.home() / "Desktop" / "output_folder_no_resample"
Path(directory_out_no_resample).mkdir(parents=True, exist_ok=True)          

#Set py_patient_file
py_patient_file=Path.home() / "Desktop" / "output_folder_no_resample"/"patient_py.xlsx"

##### Settings #####

In [100]:
#Information of the patient
image_modality = "CT"
rt_kind="DCM_RS"
list_roi=["GTV-1"]
ID_problems = []

In [101]:
df_py = info_dcm.find_ct_info(directory_dcm_out, directory_out_no_resample,image_modality,py_patient_file)
print(df_py)  

The dataframe with all headers information is in:  C:\Users\volon\Desktop\output_folder_no_resample\patient_py.xlsx
   PatientID PatientName     PatientAge  VoxelSpacingX  VoxelSpacingY  \
0          1        PAT1  Non_calcolato          0.977          0.977   

   VoxelSpacingZ                                               Path  
0              3  C:\Users\volon\Desktop\Densitometry\tutorials\...  


In [102]:
#Choose if you want to run in parallel
flag_parallel=True


#Set the number of jobs
if flag_parallel==True:
    N_jobs=2
else:
    N_jobs=1

##### Count ROIs #####

In [103]:
save_ROI = True
show_info_all = True

In [104]:
if save_ROI:
    #if True create db with all patient's ROI and relative counts
    df_ROI, df_counts = ROI.all_ROI(df_py,ID_problems,directory_out_no_resample,rt_kind)    
    print(f"dataframe ROIs:{df_ROI}")   
    print(f"dataframe counts:{df_counts}")       
    print("")
else:
    print("You chose to not extract all patients ROIs.") 
    print("")


I read the dataframe with all the ROIs of each patient


I read the dataframe with the ROI counts for each patient

dataframe ROIs:   Unnamed: 0      0
0           1  GTV-1
dataframe counts:  Counts  count
0  GTV-1      1



##### Extraction #####

In [105]:
#Extract the desired information
if show_info_all:
        
    print("Analyzing total histograms is set on: ", show_info_all)
            
    dir_files_fin = tot.no_res_and_create_histo(df_py, ID_problems, rt_kind,list_roi,directory_out_no_resample,show_info_all,save_all,N_jobs)
    print("")
                
else:
    print("You preferred to not analyze the histograms.")
    print("")  
            

Analyzing total histograms is set on:  True
The showing variable is set on:  True
The saving variable is set on:  True
All ROI's densitometric features are in C:\Users\volon\Desktop\output_folder_no_resample\Total_ROI\Histo_total_stats.xlsx

There are no problems



## **Resampling example** ##

##### Define paths and directories #####

In [106]:
#Set the directory dcm out and directory out
directory_dcm_out = Path.home() / "Desktop" / "Densitometry" / "tutorials"/"tutorial_patient"/"IBSI1_CT_phantom"

    
directory_out_resample = Path.home() / "Desktop" / "output_folder_resample"
Path(directory_out_resample).mkdir(parents=True, exist_ok=True)          

#Set py_patient_file
py_patient_file=Path.home() / "Desktop" / "output_folder_resample"/"patient_py.xlsx"

##### Settings #####

In [107]:
#Information of the patient
image_modality = "CT"
rt_kind="DCM_RS"
list_roi=["GTV-1"]
ID_problems = []

In [108]:
df_py = info_dcm.find_ct_info(directory_dcm_out, directory_out_resample,image_modality,py_patient_file)
print(df_py)  

The dataframe with all headers information is in:  C:\Users\volon\Desktop\output_folder_resample\patient_py.xlsx
   PatientID PatientName     PatientAge  VoxelSpacingX  VoxelSpacingY  \
0          1        PAT1  Non_calcolato          0.977          0.977   

   VoxelSpacingZ                                               Path  
0              3  C:\Users\volon\Desktop\Densitometry\tutorials\...  


In [109]:
#Choose if you want to run in parallel
flag_parallel=True


#Set the number of jobs
if flag_parallel==True:
    N_jobs=2
else:
    N_jobs=1

##### New Voxel spacing #####

In [110]:
#Choose if you want to resample

flag_resampling=True

if flag_resampling:
        
    #Set the new spacing criterion
    flag_new_spacing="manual"
    
    #Set the resampler
    resampler = sitk.sitkBSpline
    
    

In [111]:
#Find the new voxel spacing

#Find the new voxel spacing from global information
if flag_new_spacing=="min_global" or flag_new_spacing=="mean_global" or flag_new_spacing=="max_global":
    
    new_sp=sp.find_global_scale(df_py,flag_new_spacing)
    print("")
  
#Find the new voxel spacing from frequency-based approach
if flag_new_spacing=="frequency":
    
    save_sp = True
    
    if "y" in save_sp.lower():
        #if save=True create voxel spacing distribution for each dimension
        new_x, new_y, new_z = sp.read_spacing(df_py, directory_out_resample, save_sp=True)
        print("")
    else:
        #if save=True create voxel spacing distribution for each dimension
        new_x, new_y, new_z = sp.read_spacing(df_py, directory_out_resample, save_sp=False)
        print("")
        
        
    new_x, new_y, new_z = sp.read_spacing(df_py, directory_out_resample, save_sp) 
    new_sp = np.array([new_x, new_y, new_z])
    
    
#Set manually the new voxel spacing    
if flag_new_spacing=="manual":

    new_sp = np.array([1, 1, 3])

In [112]:
print("The new voxel spacing is: ", new_sp)
print("")

The new voxel spacing is:  [1 1 3]



##### Count ROIs #####

In [113]:
show_info_all = True
save_all = True

In [114]:
save_ROI = True

if save_ROI:
    #if True create db with all patient's ROI and relative counts
    df_ROI, df_counts = ROI.all_ROI(df_py, ID_problems,directory_out_resample,rt_kind)  
    print(f"dataframe ROIs:{df_ROI}")   
    print(f"dataframe counts:{df_counts}")     
    print("")
else:
    print("You chose to not extract all patients ROIs.") 
    print("")


I read the dataframe with all the ROIs of each patient


I read the dataframe with the ROI counts for each patient

dataframe ROIs:   Unnamed: 0      0
0           1  GTV-1
dataframe counts:  Counts  count
0  GTV-1      1



##### Extraction #####

In [115]:
#Perform resampling and extract the desired information    
if show_info_all:
    print("Analyzing total histograms is set on: ", show_info_all)
    dir_files_fin = tot.res_and_create_histo(df_py, ID_problems, new_sp, rt_kind,list_roi,directory_out_resample,show_info_all,save_all,N_jobs,resampler)
    print(dir_files_fin)
    print("")
            
else:
    print("You preferred to not analyze the histograms.")
    print("")  
        

Analyzing total histograms is set on:  True
The showing variable is set on:  True
The saving variable is set on:  True
All ROI's densitometric features are in C:\Users\volon\Desktop\output_folder_resample\Total_ROI\Histo_total_stats.xlsx

There are no problems
C:\Users\volon\Desktop\output_folder_resample\Total_ROI\Files_ok

